In [1]:
pip install transformers datasets seqeval -q

In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from seqeval.metrics import f1_score

In [3]:
dataset = load_dataset("lhoestq/conll2003")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [4]:
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [5]:
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [7]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(examples["ner_tags"][i][word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [8]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [9]:
small_train = tokenized_dataset["train"].select(range(3000))
small_val = tokenized_dataset["validation"].select(range(1000))

In [10]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50
)

In [12]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [13]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        temp_preds = []
        temp_labels = []

        for p, l in zip(pred, lab):
            if l != -100:
                temp_preds.append(label_list[p])
                temp_labels.append(label_list[l])

        true_predictions.append(temp_preds)
        true_labels.append(temp_labels)

    return {
        "f1": f1_score(true_labels, true_predictions)
    }

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

Step,Training Loss
50,0.835238
100,0.296958
150,0.177070
200,0.155304
250,0.101584


Step,Training Loss
50,0.835238
100,0.296958
150,0.177070
200,0.155304
250,0.101584
300,0.101008
350,0.094217


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=376, training_loss=0.24112929308668096, metrics={'train_runtime': 315.294, 'train_samples_per_second': 19.03, 'train_steps_per_second': 1.193, 'total_flos': 784017819648000.0, 'train_loss': 0.24112929308668096, 'epoch': 2.0})

In [16]:
trainer.evaluate()

{'eval_loss': 0.09596594423055649,
 'eval_f1': 0.8586533110740123,
 'eval_runtime': 18.4059,
 'eval_samples_per_second': 54.33,
 'eval_steps_per_second': 3.423,
 'epoch': 2.0}

In [ ]:
import torch

sentence = "John works at Google in California"

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Tokenize
inputs = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)

# Move inputs to same device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Prediction
with torch.no_grad():
    outputs = model(**inputs).logits

predictions = torch.argmax(outputs, dim=2)

predicted_labels = [label_list[p] for p in predictions[0].cpu().numpy()]

list(zip(sentence.split(), predicted_labels))

In [18]:
model.save_pretrained("model")
tokenizer.save_pretrained("model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('model/tokenizer_config.json', 'model/tokenizer.json')